In [1]:
import os 
os.getcwd()

'/home/leostre/Рабочий стол/py-boost/ablations'

In [2]:
os.chdir('..')

cifar10

In [3]:
from experiments.data.load_data import load_cifar10
import numpy as np 
from sklearn.preprocessing import LabelBinarizer

cifar10 = load_cifar10()

xtr, ytr, xte, yte, _ = zip(*cifar10)
xtr = np.concat(xtr)
ytr = LabelBinarizer().fit_transform(np.concat(ytr))
xte = np.concat(xte)
yte = LabelBinarizer().fit_transform(np.concat(yte))


BASE_GRID = {
    'subsample': [0.05, 25, .5, .75, 1.],
    'sketch_outputs': [1, 5, 10]
}

HB_GRID = {**BASE_GRID, 
    'smoothing_alpha': [0.8, 0.9, 0.95],
    'stabilization_threshold': [.5, 1., 2.]           
}

mediamill

In [68]:
# from experiments.data.load_data import load_mediamill
# mm = load_mediamill()
# X = mm['features']
# y = mm['target']

# from sklearn.model_selection import train_test_split 

# xtr, xte, ytr, yte = train_test_split(X, y, random_state=1, test_size=.4)

# BASE_GRID = {
#     'subsample': [0.05, 25, .5, .75, 1.],
#     'sketch_outputs': [10, 25, 50, 75, 101]
# }

# HB_GRID = {**BASE_GRID, 
#     'smoothing_alpha': [0.8, 0.9, 0.95],
#     'stabilization_threshold': [.5, 1., 2.]           
# }

mnist

In [9]:
from experiments.data.load_data import load_mnist
import numpy as np 
from sklearn.preprocessing import LabelBinarizer



xtr, ytr, xte, yte, _ = zip(*load_mnist())
xtr = np.concat(xtr)
ytr = (np.concat(ytr))
xte = np.concat(xte)
yte = (np.concat(yte))


BASE_GRID = {
    'subsample': [0.05, .5, 1.],
    'sketch_outputs': [1, 5, 10]
}

HB_GRID = {**BASE_GRID, 
    'smoothing_alpha': [
        # 0.8, 
        0.9, 
        # 0.95
        ],
    'stabilization_threshold': [
        .5, 1.,
                                #  2.
                                 ]           
}

In [15]:
from py_boost.gpu.sketch_boost import SketchBoost 
from functools import partial
from py_boost.gpu.history_boosting import HistoryBasedBoostingModel
from py_boost.gpu.accumulation.history_callback import WeightedHistorySampling
from experiments.models import HyperbolicWeightedHistorySampling

defaults = dict(
   loss='multilabel',
   metric='bce', 
   ntrees=10_000,
   use_hess=True, 
   es=15,
   seed=1,
   lr=0.05,
   sketch_method='topk',
)

sb = partial(SketchBoost, **defaults) 
hb = partial(HistoryBasedBoostingModel, ** defaults, multioutput_sketch=HyperbolicWeightedHistorySampling)
shb = partial(HistoryBasedBoostingModel, **defaults, multioutput_sketch=WeightedHistorySampling)


In [14]:
from sklearn.metrics import accuracy_score

from experiments.core.model_timing import GPUTimer



def count_tree_nodes_and_leaves(ensemble):
    """
    Count nodes and leaves for each tree in the ensemble.
    
    Args:
        ensemble: Ensemble object with trained trees
    
    Returns:
        dict: Dictionary containing statistics
    """
    per_tree_nodes = []
    per_tree_leaves = []
    per_group_nodes = []
    per_group_leaves = []
    
    for tree_idx, tree in enumerate(ensemble.models):
        tree_nodes = 0
        tree_leaves = 0
        group_nodes = []
        group_leaves = []
        
        # Count for each group (subtree)
        for group in range(tree.ngroups):
            node_count = 0
            leaf_count = 0
            
            # Method 1: Using test_format (after reformatting) - most reliable
            if tree.test_format is not None and tree.test_format_offsets is not None:
                # Convert to integer scalars
                offset = int(tree.test_format_offsets[group])
                
                if group + 1 < len(tree.test_format_offsets):
                    next_offset = int(tree.test_format_offsets[group + 1])
                else:
                    # Calculate from test_format length
                    next_offset = len(tree.test_format) // 4
                
                # Iterate through nodes in this group's subtree
                for node_idx in range(offset, next_offset):
                    # Each node is 4 consecutive values
                    feature_val = tree.test_format[node_idx * 4]
                    left_child = tree.test_format[node_idx * 4 + 2]
                    right_child = tree.test_format[node_idx * 4 + 3]
                    
                    if feature_val != 0:  # Valid node
                        node_count += 1
                        # Check if this is a leaf (both children negative)
                        if left_child < 0 and right_child < 0:
                            leaf_count += 1
            
            # Method 2: Using feats array (works before reformatting or if test_format is None)
            elif tree.feats is not None:
                # Non-leaf nodes: where feats >= 0
                non_leaf_mask = tree.feats[group] >= 0
                non_leaf_count = int(non_leaf_mask.sum())
                
                if hasattr(tree, 'split') and tree.split is not None:
                    # Traverse to find actual nodes
                    visited = set()
                    
                    def traverse(node_idx):
                        if node_idx >= tree.max_nodes or node_idx in visited:
                            return
                        visited.add(node_idx)
                        if tree.feats[group][node_idx] >= 0:  # Non-leaf node
                            left = int(tree.split[group][node_idx][0])
                            right = int(tree.split[group][node_idx][1])
                            if left >= 0:
                                traverse(left)
                            if right >= 0:
                                traverse(right)
                    
                    # Start traversal from root (node 0)
                    if tree.feats[group][0] >= -1:  # Valid root
                        traverse(0)
                    
                    node_count = len(visited)
                    leaf_count = sum(1 for node in visited if tree.feats[group][node] == -1)
                else:
                    # Fallback estimate
                    node_count = non_leaf_count + (non_leaf_count + 1)
                    leaf_count = non_leaf_count + 1
            else:
                # Tree not available
                node_count = 0
                leaf_count = 0
            
            group_nodes.append(node_count)
            group_leaves.append(leaf_count)
            tree_nodes += node_count
            tree_leaves += leaf_count
        
        per_tree_nodes.append(tree_nodes)
        per_tree_leaves.append(tree_leaves)
        per_group_nodes.append(group_nodes)
        per_group_leaves.append(group_leaves)
    
    return {
        'per_tree_nodes': per_tree_nodes,
        'per_tree_leaves': per_tree_leaves,
        'per_group_nodes': per_group_nodes,
        'per_group_leaves': per_group_leaves,
        'total_nodes': sum(per_tree_nodes),
        'total_leaves': sum(per_tree_leaves),
        'num_trees': len(ensemble.models)
    }


def evaluate_model(model, xte, yte, thrs=0.5):
    structure_metrics = count_tree_nodes_and_leaves(model)
    probas = model.predict(xte)
    preds = (probas > thrs).astype(int)
    humming_loss = 1 - accuracy_score(yte, preds)
    return structure_metrics | {'humming_loss': humming_loss}

from itertools import product
import pandas as pd 
from functools import reduce

import numpy as np
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def optimize_threshold_per_label(y_true, y_prob, metric='f1'):
    """
    Оптимизация порога для каждой метки
    
    Parameters:
    -----------
    y_true : array-like, shape (n_samples, n_labels)
    y_prob : array-like, shape (n_samples, n_labels)
    metric : str, 'f1' or 'f2' or 'f05'
    
    Returns:
    --------
    thresholds : array, shape (n_labels,)
    """
    n_labels = y_true.shape[1]
    thresholds = []
    
    for i in range(n_labels):
        # Получаем precision, recall и пороги для текущей метки
        precision, recall, thresh = precision_recall_curve(y_true[:, i], y_prob[:, i])
        
        # Вычисляем F1 для каждого порога
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
        
        # Находим порог, максимизирующий F1
        best_idx = np.argmax(f1_scores)
        
        # В precision_recall_curve последний порог = 1.0
        if best_idx < len(thresh):
            best_thresh = thresh[best_idx]
        else:
            best_thresh = 1.0
            
        thresholds.append(best_thresh)
    
    return np.array(thresholds)


from tqdm import tqdm 
import cupy as cp 

def run_experiment(model_template, xtr, ytr, xte, yte, grid, n_repeats, dump_file, skip_first=True):
    if os.path.exists(dump_file):
        res = pd.read_csv(dump_file)
    else:
        res = pd.DataFrame(columns=[*grid.keys(), 'iter', 'inference_time', 'train_time', 'per_tree_nodes', 'per_group_nodes', 'per_tree_leaves', 'per_group_leaves', 'total_nodes', 'total_leaves', 'num_trees'])
    skip_first = res.shape[0] if skip_first else 0
    total = reduce(int.__mul__, [len(x) for x in grid.values()]) * n_repeats

    eval_x, _, eval_y, _ = train_test_split(xtr, ytr, random_state=1, train_size=.1)
    # eval_x, eval_y = xtr, ytr
    # xtr = cp.asarray(xtr)
    # ytr = cp.asarray(ytr)
    # xte = cp.asarray(xte)
    # yte = cp.asarray(yte)
    try:
        for i, hp in tqdm(enumerate(product(*grid.values())), total=total, desc='combo: '):
            try:
                if skip_first and i < skip_first:
                    continue
                param_dict = dict(zip(grid.keys(), hp))
                for iteration in range(n_repeats):
                    model = model_template(**param_dict)
                    with GPUTimer() as train_time:
                        model.fit(xtr, ytr, eval_sets=[{'X': eval_x, 'y': eval_y}])

                    with GPUTimer() as inf_time:
                        preds_tr = model.predict(xtr)
                    thrs = optimize_threshold_per_label(ytr, preds_tr)
                    result = evaluate_model(model, xte, yte, thrs[None])
                    record = param_dict | result | {'iter': iteration, 'train_time': train_time.time, 'inference_time': inf_time.time}
                    res.loc[res.shape[0]] = record
            except KeyboardInterrupt:
                raise
            except Exception as x:
                import traceback
                print(traceback.format_exc())
                print(f'Config {hp} failed due to: {x}')

    except KeyboardInterrupt:
        print('Stopped at iter:', i) 
    finally:
        res.to_csv(dump_file)
        



In [18]:
from pathlib import Path

NNODES_BASELINE = Path('/home/leostre/Рабочий стол/py-boost/ablations/nnodes/sketch_boost.csv') 
NNODES_HYPER = Path('/home/leostre/Рабочий стол/py-boost/ablations/nnodes/HypHASBoost.csv') 
NNODES_SIG = Path('/home/leostre/Рабочий стол/py-boost/ablations/nnodes/SigHASBoost.csv') 

In [ ]:
run_experiment(sb, xtr, ytr, xte, yte, BASE_GRID, 3, NNODES_BASELINE)

In [13]:
run_experiment(hb, xtr, ytr, xte, yte, HB_GRID, 3, NNODES_HYPER)

combo:   0%|          | 0/54 [00:00<?, ?it/s]

[12:52:11] Stdout logging level is INFO.
[12:52:11] GDBT train starts. Max iter 10000, early stopping rounds 15
[12:52:11] Iter 0; Sample 0, BCE = 0.3087013357885884; 
[12:52:12] Iter 10; Sample 0, BCE = 0.2309679894533608; 
[12:52:12] Iter 20; Sample 0, BCE = 0.19087171481246346; 
[12:52:12] Iter 30; Sample 0, BCE = 0.1646269347811979; 
[12:52:12] Iter 40; Sample 0, BCE = 0.1469024477087565; 
[12:52:12] Iter 50; Sample 0, BCE = 0.13339151555887252; 
[12:52:12] Iter 60; Sample 0, BCE = 0.12251433030806952; 
[12:52:12] Iter 70; Sample 0, BCE = 0.11417898915594517; 
[12:52:12] Iter 80; Sample 0, BCE = 0.10732850270247352; 
[12:52:12] Iter 90; Sample 0, BCE = 0.10158447382181819; 
[12:52:12] Iter 100; Sample 0, BCE = 0.09734041945440744; 
[12:52:12] Iter 110; Sample 0, BCE = 0.09305549781418612; 
[12:52:12] Iter 120; Sample 0, BCE = 0.0895712725756451; 
[12:52:13] Iter 130; Sample 0, BCE = 0.08649080977164639; 
[12:52:13] Iter 140; Sample 0, BCE = 0.08360468820756578; 
[12:52:13] Iter 150

combo:   2%|▏         | 1/54 [05:46<5:05:38, 346.00s/it]

[12:57:58] Stdout logging level is INFO.
[12:57:58] GDBT train starts. Max iter 10000, early stopping rounds 15
[12:57:58] Iter 0; Sample 0, BCE = 0.30870133614856343; 
[12:57:58] Iter 10; Sample 0, BCE = 0.2309962722774046; 
[12:57:58] Iter 20; Sample 0, BCE = 0.19088600651060722; 
[12:57:58] Iter 30; Sample 0, BCE = 0.16465318362360506; 
[12:57:58] Iter 40; Sample 0, BCE = 0.14710185092700703; 
[12:57:58] Iter 50; Sample 0, BCE = 0.1334204361513421; 
[12:57:58] Iter 60; Sample 0, BCE = 0.12244867894017664; 
[12:57:58] Iter 70; Sample 0, BCE = 0.11393340391692826; 
[12:57:58] Iter 80; Sample 0, BCE = 0.10704097522748163; 
[12:57:58] Iter 90; Sample 0, BCE = 0.10107800042725427; 
[12:57:58] Iter 100; Sample 0, BCE = 0.09646209763623814; 
[12:57:58] Iter 110; Sample 0, BCE = 0.09227301576421823; 
[12:57:59] Iter 120; Sample 0, BCE = 0.08886472276822746; 
[12:57:59] Iter 130; Sample 0, BCE = 0.08522194687852296; 
[12:57:59] Iter 140; Sample 0, BCE = 0.08226647278913603; 
[12:57:59] Iter 

combo:   4%|▎         | 2/54 [11:32<5:00:16, 346.47s/it]

...

combo:  30%|██▉       | 16/54 [1:40:16<4:26:48, 421.26s/it]

[14:32:28] Stdout logging level is INFO.
[14:32:28] GDBT train starts. Max iter 10000, early stopping rounds 15
[14:32:28] Iter 0; Sample 0, BCE = 0.3071447143984222; 
[14:32:28] Iter 10; Sample 0, BCE = 0.21660322578519536; 
[14:32:28] Iter 20; Sample 0, BCE = 0.1740750953062445; 
[14:32:28] Iter 30; Sample 0, BCE = 0.14534720266598675; 
[14:32:28] Iter 40; Sample 0, BCE = 0.12446110566001546; 
[14:32:28] Iter 50; Sample 0, BCE = 0.10922692632177376; 
[14:32:28] Iter 60; Sample 0, BCE = 0.0970497645273233; 
[14:32:28] Iter 70; Sample 0, BCE = 0.08738082466798677; 
[14:32:29] Iter 80; Sample 0, BCE = 0.07947005514686578; 
[14:32:29] Iter 90; Sample 0, BCE = 0.07291752752287477; 
[14:32:29] Iter 100; Sample 0, BCE = 0.06736143179645192; 
[14:32:29] Iter 110; Sample 0, BCE = 0.06263512382943948; 
[14:32:29] Iter 120; Sample 0, BCE = 0.05859707995077893; 
[14:32:29] Iter 130; Sample 0, BCE = 0.05491036182436203; 
[14:32:29] Iter 140; Sample 0, BCE = 0.05177329470828665; 
[14:32:29] Iter 1

combo:  31%|███▏      | 17/54 [1:47:41<4:24:13, 428.47s/it]

[14:39:53] Stdout logging level is INFO.
[14:39:53] GDBT train starts. Max iter 10000, early stopping rounds 15
[14:39:53] Iter 0; Sample 0, BCE = 0.3071447154074928; 
[14:39:53] Iter 10; Sample 0, BCE = 0.21660322812780922; 
[14:39:53] Iter 20; Sample 0, BCE = 0.1740750985982251; 
[14:39:53] Iter 30; Sample 0, BCE = 0.14534720420219252; 
[14:39:53] Iter 40; Sample 0, BCE = 0.12446110585728389; 
[14:39:54] Iter 50; Sample 0, BCE = 0.1092269266085213; 
[14:39:54] Iter 60; Sample 0, BCE = 0.09704976451041698; 
[14:39:54] Iter 70; Sample 0, BCE = 0.08738082460960873; 
[14:39:54] Iter 80; Sample 0, BCE = 0.07947005487274514; 
[14:39:54] Iter 90; Sample 0, BCE = 0.07291752742675364; 
[14:39:54] Iter 100; Sample 0, BCE = 0.0673614319027987; 
[14:39:54] Iter 110; Sample 0, BCE = 0.06263512392731016; 
[14:39:54] Iter 120; Sample 0, BCE = 0.05859708010619256; 
[14:39:54] Iter 130; Sample 0, BCE = 0.05491036199717301; 
[14:39:55] Iter 140; Sample 0, BCE = 0.05177329483512992; 
[14:39:55] Iter 15

combo:  33%|███▎      | 18/54 [1:55:03<3:50:07, 383.54s/it]


In [19]:
run_experiment(shb, xtr, ytr, xte, yte, HB_GRID, 3, NNODES_SIG)

combo:   0%|          | 0/54 [00:00<?, ?it/s]

[17:33:09] Stdout logging level is INFO.
[17:33:09] GDBT train starts. Max iter 10000, early stopping rounds 15
[17:33:09] Iter 0; Sample 0, BCE = 0.3087013359433423; 
[17:33:09] Iter 10; Sample 0, BCE = 0.2309710229319779; 
[17:33:09] Iter 20; Sample 0, BCE = 0.1908874242944526; 
[17:33:09] Iter 30; Sample 0, BCE = 0.16461041817349936; 
[17:33:09] Iter 40; Sample 0, BCE = 0.1470526261160754; 
[17:33:09] Iter 50; Sample 0, BCE = 0.13330831016772987; 
[17:33:09] Iter 60; Sample 0, BCE = 0.12239612045636146; 
[17:33:09] Iter 70; Sample 0, BCE = 0.11391371187922532; 
[17:33:09] Iter 80; Sample 0, BCE = 0.10725142741154418; 
[17:33:09] Iter 90; Sample 0, BCE = 0.10124932722906631; 
[17:33:09] Iter 100; Sample 0, BCE = 0.09656373657593206; 
[17:33:09] Iter 110; Sample 0, BCE = 0.09238167871920992; 
[17:33:10] Iter 120; Sample 0, BCE = 0.08874281686815773; 
[17:33:10] Iter 130; Sample 0, BCE = 0.0855662855032257; 
[17:33:10] Iter 140; Sample 0, BCE = 0.08265222320082044; 
[17:33:10] Iter 150

combo:   2%|▏         | 1/54 [05:55<5:13:59, 355.46s/it]

[17:39:04] Stdout logging level is INFO.
[17:39:04] GDBT train starts. Max iter 10000, early stopping rounds 15
[17:39:04] Iter 0; Sample 0, BCE = 0.30870133589088; 
[17:39:04] Iter 10; Sample 0, BCE = 0.23097836139510974; 
[17:39:04] Iter 20; Sample 0, BCE = 0.19082154293370474; 
[17:39:04] Iter 30; Sample 0, BCE = 0.16456121933580486; 
[17:39:04] Iter 40; Sample 0, BCE = 0.1469707113167337; 
[17:39:05] Iter 50; Sample 0, BCE = 0.13335568779900359; 
[17:39:05] Iter 60; Sample 0, BCE = 0.12233475992113778; 
[17:39:05] Iter 70; Sample 0, BCE = 0.11378954058681169; 
[17:39:05] Iter 80; Sample 0, BCE = 0.1068974362377385; 
[17:39:05] Iter 90; Sample 0, BCE = 0.10101305162876233; 
[17:39:05] Iter 100; Sample 0, BCE = 0.09660273494361282; 
[17:39:05] Iter 110; Sample 0, BCE = 0.09261302378148178; 
[17:39:05] Iter 120; Sample 0, BCE = 0.08904466856140733; 
[17:39:05] Iter 130; Sample 0, BCE = 0.08561732406620086; 
[17:39:05] Iter 140; Sample 0, BCE = 0.08283264471632018; 
[17:39:05] Iter 150

combo:   4%|▎         | 2/54 [12:09<5:17:43, 366.61s/it]

...

combo:  30%|██▉       | 16/54 [1:37:48<4:10:58, 396.27s/it]

[19:10:57] Stdout logging level is INFO.
[19:10:57] GDBT train starts. Max iter 10000, early stopping rounds 15
[19:10:57] Iter 0; Sample 0, BCE = 0.30714471489041795; 
[19:10:58] Iter 10; Sample 0, BCE = 0.21660322726873327; 
[19:10:58] Iter 20; Sample 0, BCE = 0.1740750983243542; 
[19:10:58] Iter 30; Sample 0, BCE = 0.14534720408672666; 
[19:10:58] Iter 40; Sample 0, BCE = 0.12446110607661713; 
[19:10:58] Iter 50; Sample 0, BCE = 0.10922692671402638; 
[19:10:58] Iter 60; Sample 0, BCE = 0.0970497647492111; 
[19:10:58] Iter 70; Sample 0, BCE = 0.08738082508431738; 
[19:10:58] Iter 80; Sample 0, BCE = 0.07947005542388262; 
[19:10:58] Iter 90; Sample 0, BCE = 0.07291752771702627; 
[19:10:58] Iter 100; Sample 0, BCE = 0.06736143221090041; 
[19:10:59] Iter 110; Sample 0, BCE = 0.06263512430147418; 
[19:10:59] Iter 120; Sample 0, BCE = 0.058597080423225804; 
[19:10:59] Iter 130; Sample 0, BCE = 0.0549103622483421; 
[19:10:59] Iter 140; Sample 0, BCE = 0.05177329520412263; 
[19:10:59] Iter 

combo:  31%|███▏      | 17/54 [1:44:40<4:07:11, 400.85s/it]

[19:17:49] Stdout logging level is INFO.
[19:17:49] GDBT train starts. Max iter 10000, early stopping rounds 15
[19:17:49] Iter 0; Sample 0, BCE = 0.30714471506163765; 
[19:17:49] Iter 10; Sample 0, BCE = 0.21660322927616407; 
[19:17:49] Iter 20; Sample 0, BCE = 0.1740750980098207; 
[19:17:49] Iter 30; Sample 0, BCE = 0.14534720482757302; 
[19:17:49] Iter 40; Sample 0, BCE = 0.12446110643783832; 
[19:17:49] Iter 50; Sample 0, BCE = 0.10922692694547526; 
[19:17:50] Iter 60; Sample 0, BCE = 0.09704976528454375; 
[19:17:50] Iter 70; Sample 0, BCE = 0.08738082517260543; 
[19:17:50] Iter 80; Sample 0, BCE = 0.07947005551919659; 
[19:17:50] Iter 90; Sample 0, BCE = 0.07291752784548111; 
[19:17:50] Iter 100; Sample 0, BCE = 0.06736143243376054; 
[19:17:50] Iter 110; Sample 0, BCE = 0.06263512447366841; 
[19:17:50] Iter 120; Sample 0, BCE = 0.05859708064467643; 
[19:17:50] Iter 130; Sample 0, BCE = 0.05491036254120004; 
[19:17:50] Iter 140; Sample 0, BCE = 0.05177329543398079; 
[19:17:51] Iter

combo:  33%|███▎      | 18/54 [1:51:32<3:43:05, 371.81s/it]


In [82]:
sb_.fit(xtr, ytr)

[17:29:51] Stdout logging level is INFO.
[17:29:51] GDBT train starts. Max iter 10, early stopping rounds 15
[17:29:51] Iter 0; 
[17:29:51] Iter 9; 


In [87]:
count_tree_nodes_and_leaves(sb_)

{'per_tree_nodes': [35, 40, 55, 55, 54, 52, 41, 54, 61, 54],
 'per_tree_leaves': [13, 16, 26, 26, 26, 25, 19, 26, 31, 26],
 'per_group_nodes': [[35],
  [40],
  [55],
  [55],
  [54],
  [52],
  [41],
  [54],
  [61],
  [54]],
 'per_group_leaves': [[13],
  [16],
  [26],
  [26],
  [26],
  [25],
  [19],
  [26],
  [31],
  [26]],
 'total_nodes': 501,
 'total_leaves': 234,
 'num_trees': 10}